In [26]:
import os

from bs4 import BeautifulSoup
from dotenv import load_dotenv
from openai import OpenAI
import requests


In [6]:
load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")

anthropic_api_url = "https://api.anthropic.com/v1/"

openai = OpenAI(api_key=openai_api_key)
anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_api_url)

In [63]:
def fetch_website_contents(url: str) -> str | None:
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
    }
    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        return None
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    return title + "\n\n" + text

In [64]:
def fetch_wikipedia_page(subject:str) -> str | None:
    url = f"https://en.wikipedia.org/wiki/{subject.lower().strip().replace(' ', '_').replace("-", "_")}"
    return fetch_website_contents(url)

In [66]:
def summarize_text(text: str, model: str = "gpt-4o-mini") -> str | None:
    system_prompt = """You are a helpful assistant that summarizes text in a concise, structured and compelling way,
    ignoring text that might be navigation related. 
    Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown."""
    user_prompt = f"Summarize the following text in a concise, structured and compelling way:\n\n{text}"
    response = openai.chat.completions.create(
        model=model,
        messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}],
        temperature=0.7,
    )
    return response.choices[0].message.content

In [ ]:
def find_out_the_main_spoken_language_in_city(city: str, model: str = "gpt-4o-mini") -> str | None:
    system_prompt = """You are a helpful assistant that finds out the main language spoken in a city.
    Respond with the name of the language only, without any additional text."""
    user_prompt = f"What is the main language spoken in {city}?"
    response = openai.chat.completions.create(
        model=model,
        messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}],
        temperature=0,
    )
    return response.choices[0].message.content

In [72]:
language = find_out_the_main_spoken_language_in_city("Tokyo")
language

'The main language spoken in Tokyo is Japanese.'